# Evaluate Model 2 (conditions) t5-small: 300k-trained vs 41k-trained, same held-out test (Kaggle GPU)

Both checkpoints scored on the **same** leak-free 300k-pool `conditions_test.jsonl`
(7,714 rows), top-1/top-5 beam search, so the only difference is training-data volume:

- **NEW**: t5-small trained on 138,869 conditions rows (300k pool). Mounted from this
  project's training-kernel output.
- **OLD**: t5-small trained on 41,139 rows (60k pool), the current RESULTS.md Model 2.
  Mounted from an uploaded dataset.

Local eval kept dying to session restarts; running it on Kaggle GPU is both faster and
independent of the local session.

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, os
print("/kaggle/input contents:", os.listdir("/kaggle/input"))
cfgs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("config.json found:")
for c in cfgs: print("  ", c)

test_file = next(f for f in glob.glob("/kaggle/input/**/conditions_test.jsonl", recursive=True))
new_dir = os.path.dirname(next(c for c in cfgs if "t5small-300k-ckpt" in c))
old_dir = os.path.dirname(next(c for c in cfgs if "t5small-41k" in c))
print("test:", test_file, "rows", sum(1 for _ in open(test_file)))
print("NEW:", new_dir)
print("OLD:", old_dir)

In [ ]:
# NEW (300k-trained) on the shared held-out test
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{new_dir}" --test-file "{test_file}" \
    --num-beams 10 --device cuda --batch-size 64 \
    --output /kaggle/working/eval_new_300k.json
print("NEW done")

In [ ]:
# OLD (41k-trained) on the SAME test
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{old_dir}" --test-file "{test_file}" \
    --num-beams 10 --device cuda --batch-size 64 \
    --output /kaggle/working/eval_old_41k.json
print("OLD done")

In [ ]:
import json
for tag,f in [("NEW 300k","/kaggle/working/eval_new_300k.json"),("OLD 41k","/kaggle/working/eval_old_41k.json")]:
    print("===",tag,"===")
    print(json.dumps(json.load(open(f))["summary"], indent=2))